# Task 4 — Step 2: Manual Audit & Safety Calibration Analysis

### Research Questions & Key Metrics
1. **Safety Calibration Frontier**:
   - Safe-prompt over-refusal rate: $\Pr[\text{OVER\_REFUSAL} | \text{safe prompt}]$ (exaggerated safety).
   - Unsafe-prompt compliance rate: $\Pr[\text{UNSAFE\_COMPLIANCE} | \text{unsafe prompt}]$ (harmful failure).
   - *A single refusal score conflates these two! Both must be reported.*
2. **Human vs. AI Judge Audit Agreement**:
   - Evaluate agreement on the fixed 60-example subset balanced across safe/unsafe prompts.
   - Characterize AI judge failure modes (misclassifying figurative language, sarcasm, technical terminology).
3. **Category-Level Analysis**:
   - Examine distribution across XSTest categories.


In [ ]:
%matplotlib inline
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Configure matplotlib style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Ensure project root is in sys.path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
try:
    os.chdir(ROOT)
except Exception:
    pass

print("Project root:", ROOT)


## 1. Run Evaluation & Synthesis


In [ ]:
FORCE_RUN = False
if FORCE_RUN:
    !python -m task4_safety.make_audit_sheet --config configs/feedback.yaml
    !python -m task4_safety.evaluate_safety --config configs/feedback.yaml
else:
    print("[OK] Loading existing safety calibration results...")


## 2. Safety Calibration Summary Table across All 4 Policies


In [ ]:
safety_file = "results/task4_safety/safety_evaluation_results.json"
if os.path.exists(safety_file):
    with open(safety_file, encoding='utf-8') as f:
        s_data = json.load(f)
        
    rows = []
    for pol in ["sft", "dpo", "ppo", "grpo"]:
        if pol in s_data:
            st = s_data[pol]
            rows.append({
                "Policy": pol.upper(),
                "Safe Answer Rate": f"{st.get('safe_prompt_answer_rate', 0)*100:.1f}%",
                "Safe Over-Refusal Rate": f"{st.get('safe_prompt_over_refusal_rate', 0)*100:.1f}%",
                "Unsafe Compliance Rate": f"{st.get('unsafe_prompt_unsafe_compliance_rate', 0)*100:.1f}%",
                "Justified Refusal Rate": f"{st.get('unsafe_prompt_justified_refusal_rate', 0)*100:.1f}%",
                "Mean Length (tokens)": f"{st.get('mean_response_length_tokens', 0):.1f}"
            })
    df_safety = pd.DataFrame(rows)
    display(df_safety)


## 3. Human vs. AI Judge Audit Agreement (60 Examples)


In [ ]:
audit_file = "results/task4_safety/manual_audit_sheet.csv"
if os.path.exists(audit_file):
    df_audit = pd.read_csv(audit_file)
    print(f"Loaded {len(df_audit)} manually audited examples.")
    display(df_audit[['prompt_id', 'focus', 'prompt', 'model_response']].head(3))


## 4. Visualizing the Safety Calibration Frontier
Ideal calibrated policy should reside in the **bottom-left** corner (low over-refusal AND low unsafe compliance).


In [ ]:
if os.path.exists(safety_file) and rows:
    pols = [r["Policy"] for r in rows]
    over_refs = [float(r["Safe Over-Refusal Rate"].replace('%','')) for r in rows]
    unsafes = [float(r["Unsafe Compliance Rate"].replace('%','')) for r in rows]
    
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = {'SFT': 'gray', 'DPO': '#3B82F6', 'PPO': '#F59E0B', 'GRPO': '#10B981'}
    for i, p in enumerate(pols):
        ax.scatter(over_refs[i], unsafes[i], s=200, color=colors.get(p, 'purple'), zorder=5)
        ax.annotate(p, (over_refs[i] + 0.5, unsafes[i] + 0.3), fontsize=12, fontweight='bold')
        
    ax.set_xlabel('Safe Prompt Over-Refusal Rate (%) — [Harmlessness Overreach]')
    ax.set_ylabel('Unsafe Compliance Rate (%) — [Harmful Failure]')
    ax.set_title('Safety Calibration Frontier (Ideal = Bottom-Left)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


## 5. Publication Dashboard Inspection


In [ ]:
p = "report/figures/task4_safety_calibration_dashboard.png"
if os.path.exists(p):
    print(f"Displaying publication figure: {p}")
    display(Image(filename=p, width=700))


## 6. Qualitative Disagreements Analysis
Include brief examples showing:
1. **Harmful compliance**: An unsafe request that slipped through.
2. **Exaggerated refusal**: A benign query with sensitive terms (e.g. killing a process) falsely refused.
3. State whether disagreements reflected policy differences, judge errors, or both.
